In [1]:
import os
import sys
import glob
from pathlib import Path
from astropy.io import fits
import numpy as np
from numpy import savetxt
import matplotlib
import matplotlib.pyplot as plt
from PyAstronomy import pyasl
from scipy.interpolate import interp1d
from astropy import units as u
from astropy.coordinates import SkyCoord
#install dustmaps from https://github.com/gregreen/dustmaps
from dustmaps.sfd import SFDQuery
import pandas as pd
from astropy.table import Table
import gc
matplotlib.use("Agg")
plt.ioff()
from PIL import Image
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.cbook import get_sample_data
from astroquery.sdss import SDSS
import urllib.request
import warnings
from joblib import Parallel, delayed

In [3]:
storage_dir = os.path.join(os.path.expanduser("~"),'Data_storageHII/')
os.makedirs(storage_dir, exist_ok=True)

In [4]:
fits_folder = storage_dir + '/HIIG_specDR7/'
spec_list = os.listdir(fits_folder)
spec_list.remove('DOWNspec_listDR7.txt')
spec_list = sorted(spec_list)
len(spec_list)

121

In [66]:
hdu = fits.open(fits_folder+spec_list[9])
plate,mjd,fiber = int(hdu[0].header['PLATEID']),int(hdu[0].header['MJD']),int(hdu[0].header['FIBERID'])
print(plate,mjd,fiber)
result = SDSS.query_specobj(plate=plate,mjd=mjd,fiberID=fiber)
result

429 51820 495


ra,dec,objid,run,rerun,camcol,field,z,plate,mjd,fiberID,specobjid,run2d
float64,float64,uint64,int64,int64,int64,int64,float64,int64,int64,int64,uint64,int64
26.779343875627,13.9414254674982,1237653651846332519,1904,301,3,324,0.05662303,429,51820,495,483147155133982720,26


In [ ]:
26.77929 13.94144 0.05671

61

In [65]:
for i in range(len(spec_list)):
    hdu = fits.open(fits_folder+spec_list[i])
    plate,mjd,fiber = int(hdu[0].header['PLATEID']),int(hdu[0].header['MJD']),int(hdu[0].header['FIBERID'])
    #print(plate,mjd,fiber)
    hdu.close()
    result = SDSS.query_specobj(plate=plate,mjd=mjd,fiberID=fiber)

    status = str(type(result))

    tol = 0.0001

    if (status != "<class 'NoneType'>"):
        tabla5_art = pd.read_csv('Chavez2014_Tabla5.csv', sep=",", header=0) 
        f = (abs(tabla5_art['alpha(J2K)'] - result['ra'][0]) < tol) & (abs(tabla5_art['delta(J2K)'] - result['dec'][0]) < tol) & (abs(tabla5_art['z_hel'] - result['z'][0]) < tol)
        seleccion_a_index = tabla5_art[f]
        if len(seleccion_a_index) == 1:
            #re_ra,re_dec,re_z = 
            re_ra,re_dec,re_z =  round(abs(seleccion_a_index['alpha(J2K)'].iloc[0] - result['ra'][0]),4),round(abs(seleccion_a_index['delta(J2K)'].iloc[0] - result['dec'][0]),4),round(abs(seleccion_a_index['z_hel'].iloc[0] - result['z'][0]),4)
            print(i,spec_list[i],' -> ',seleccion_a_index['Index'].iloc[0],' en Tabla 5   ->   ',re_ra,'    ',re_dec,'    ',re_z)
        else:
            print(i,' Parece haber ambiguedad con espectro ',spec_list[i])
    else:
        print('Espectro ',i,spec_list[i],' no esta en el Query de SDSS')




0 spSpec-51608-0267-421.fit  ->  61  en Tabla 5   ->    0.0      0.0      0.0
1 spSpec-51690-0341-606.fit  ->  103  en Tabla 5   ->    0.0      0.0      0.0
2 spSpec-51793-0388-457.fit  ->  1  en Tabla 5   ->    0.0      0.0      0.0
3 spSpec-51810-0415-141.fit  ->  19  en Tabla 5   ->    0.0      0.0001      0.0
4 spSpec-51811-0381-370.fit  ->  126  en Tabla 5   ->    0.0001      0.0      0.0
5 spSpec-51816-0382-328.fit  ->  127  en Tabla 5   ->    0.0001      0.0      0.0
6 spSpec-51816-0410-220.fit  ->  16  en Tabla 5   ->    0.0      0.0      0.0
7 spSpec-51817-0418-302.fit  ->  5  en Tabla 5   ->    0.0      0.0      0.0001
8 spSpec-51820-0400-441.fit  ->  9  en Tabla 5   ->    0.0      0.0      0.0
9 spSpec-51820-0429-495.fit  ->  11  en Tabla 5   ->    0.0001      0.0      0.0001
10 spSpec-51821-0384-281.fit  ->  128  en Tabla 5   ->    0.0      0.0      0.0
11 spSpec-51877-0447-361.fit  ->  41  en Tabla 5   ->    0.0      0.0      0.0
12 spSpec-51882-0442-156.fit  ->  32  en Ta